Uses the [route API](https://valhalla.github.io/valhalla/api/route/api-reference/) from the [demo instance](https://valhalla.openstreetmap.de/) of [Valhalla](https://valhalla.github.io/valhalla/).


In [16]:
valhalla_params = {
    "locations": [
        {"lat": 40.7411, "lon": -73.9897},
        {"lat": 40.7308, "lon": -73.9973},
    ],
    "costing": "bicycle",
    "costing_options": {
        "bicycle": {
            "bicycle_type": "road",
            "use_roads": 0.2,
            "use_hills": 0.3,
            "use_ferry": 0.0,
            "avoid_bad_surfaces": 0.8,
        }
    },
    "directions_options": {"units": "miles"},
    "alternates": 2,
}

In [17]:
import json

import folium
import httpx
import polyline
from IPython.display import display

In [18]:
url = "https://valhalla1.openstreetmap.de/route"
request_json = json.dumps(valhalla_params, separators=(",", ":"))

with httpx.Client(timeout=60.0) as client:
    response = client.get(
        url,
        params={"json": request_json},
        headers={"Accept": "application/json"},
    )
    response.raise_for_status()

content_type = response.headers.get("content-type", "")
if "application/json" not in content_type:
    preview = response.text[:200].replace("\n", " ")
    raise ValueError(f"Expected JSON but got {content_type}: {preview}")

route_response = response.json()
if route_response.get("error"):
    raise ValueError(route_response["error"])

print(f"HTTP {response.status_code}")
print("Top-level keys:", list(route_response.keys()))

HTTP 200
Top-level keys: ['trip']


In [19]:
def extract_trips(payload):
    trips = []

    primary_trip = payload.get("trip")
    if primary_trip:
        trips.append(("Primary", primary_trip))

    for i, alt in enumerate(payload.get("alternates", []), start=1):
        alt_trip = alt.get("trip", alt)
        if alt_trip:
            trips.append((f"Alternate {i}", alt_trip))

    return trips


def decode_trip_shape(trip):
    points = []
    for leg in trip.get("legs", []):
        leg_points = polyline.decode(leg["shape"], precision=6)
        if points and leg_points and points[-1] == leg_points[0]:
            leg_points = leg_points[1:]
        points.extend(leg_points)
    return points


trips = extract_trips(route_response)
if not trips:
    raise ValueError("No routes were returned by Valhalla.")

route_points = [(label, decode_trip_shape(trip)) for label, trip in trips]
route_points = [(label, pts) for label, pts in route_points if pts]
if not route_points:
    raise ValueError("Routes were returned but no polyline geometry was found.")

center = route_points[0][1][0]
m = folium.Map(location=[center[0], center[1]], zoom_start=14, tiles="CartoDB Positron")

colors = ["#E63946", "#1D3557", "#2A9D8F", "#F4A261", "#457B9D"]
for idx, (label, points) in enumerate(route_points):
    color = colors[idx % len(colors)]
    folium.PolyLine(
        points,
        color=color,
        weight=5,
        opacity=0.85,
        tooltip=label,
    ).add_to(m)

start = route_points[0][1][0]
end = route_points[0][1][-1]
folium.Marker(start, tooltip="Start", icon=folium.Icon(color="green")).add_to(m)
folium.Marker(end, tooltip="End", icon=folium.Icon(color="red")).add_to(m)

print(f"Mapped {len(route_points)} route(s).")
display(m)

Mapped 1 route(s).
